In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import optuna
from catboost import CatBoostRegressor
import joblib
optuna.logging.set_verbosity(optuna.logging.WARNING)
import json
from sklearn.preprocessing import StandardScaler

# Functions

In [2]:
def naive_forecaster(X_test):
    lag1_cols = [f'DA_price lag1_hour{i}' for i in range(24)]
    lag7_cols = [f'DA_price lag7_hour{i}' for i in range(24)]
    prediction = []
    for index, row in X_test.iterrows():
        if pd.to_datetime(index).weekday() in [1,2,3,4,6]:
            prediction.append(row[lag1_cols].values)
        else:
            prediction.append(row[lag7_cols].values)
    prediction = np.array(prediction)
    return prediction

# calculate smape
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

#day average error
def dae(y_true, y_pred):
    return np.mean(np.abs(np.mean(y_pred,axis=1) - np.mean(y_true,axis=1)))

def rmae(y_true, y_pred, y_pred_naive):
    mae_pred = np.mean(np.abs(y_pred - y_true))
    mae_naive = np.mean(np.abs(y_pred_naive - y_true))
    return mae_pred/mae_naive

In [ ]:
def get_best_params_and_test(country, period=0, n_trials=50):

    # Load the data
    inputs = pd.read_csv(os.path.join("cut_data", country, "inputs.csv"),index_col=0)
    inputs.fillna(value=0, inplace=True)
    outputs = pd.read_csv(os.path.join("cut_data",country, "outputs.csv"),index_col=0)
    outputs.fillna(value=0, inplace=True)
    
    # fill nan with ffil
    inputs = inputs.fillna(value=0)
    #outputs = outputs.ffill()

    X_train = inputs.loc[f'{2015+period}-01-08':f'{2018+period}-12-31']
    y_train = outputs.loc[f'{2015+period}-01-08':f'{2018+period}-12-31']

    X_val = inputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']
    y_val = outputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']

    X_test = inputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']
    y_test = outputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']

    # Scaling
    scaler1, scaler2 = StandardScaler(), StandardScaler()

    X_train = pd.DataFrame(scaler1.fit_transform(X_train), columns=X_train.columns)
    X_val = pd.DataFrame(scaler1.transform(X_val), columns=X_val.columns)
    X_test = pd.DataFrame(scaler1.transform(X_test), columns=X_test.columns)

    y_train = pd.DataFrame(scaler2.fit_transform(y_train), columns=y_train.columns)
    y_val = pd.DataFrame(scaler2.transform(y_val), columns=y_val.columns)
    y_test = pd.DataFrame(scaler2.transform(y_test), columns=y_test.columns)

    def objective(trial):
        # Define the CatBoost model with hyperparameters to tune
        model = CatBoostRegressor(
            depth=trial.suggest_int('depth', 4, 8),
            learning_rate=trial.suggest_float('learning_rate', 1e-3, 1, log=True),
            iterations=trial.suggest_int('iterations', 100, 500),
            l2_leaf_reg=trial.suggest_int('l2_leaf_reg', 1, 10),
            silent=True,
            objective='MultiRMSE',
            task_type='GPU',
            use_best_model=True,
            eval_metric='MultiRMSE',
            boosting_type='Plain',
        )
    
        # Fit the model
        model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=10)
    
        # Predict on validation set
        y_pred = model.predict(X_val)
        del model
        torch.cuda.empty_cache()
    
        # Flatten the predicted and actual values
        y_pred_flattened = y_pred.flatten()
        y_val_flattened = y_val.values.flatten()
        y_pred_naive_flattened = naive_forecaster(X_val).flatten()
    
        # Calculate mean absolute error between flattened vectors
        rmae_val = rmae(y_val_flattened, y_pred_flattened, y_pred_naive_flattened)
    
        # Return the MAE as the objective to minimize
        return rmae_val
    
    # Create an Optuna study
    study = optuna.create_study(direction='minimize')
    
    # Optimize the hyperparameters
    study.optimize(objective, n_trials=n_trials)  # You can adjust the number of trials
    
    model = CatBoostRegressor(**study.best_params,
            silent=True,
            objective='MultiRMSE',
            task_type='GPU',
            use_best_model=True,
            eval_metric='MultiRMSE',
            boosting_type='Plain', random_state=42)

    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=10)

    y_pred = model.predict(X_test)

    y_pred_naive = naive_forecaster(X_test)
    y_test_val = y_test.values

    # inverse transform y_pred, y_pred_naive, y_test_val
    y_pred = scaler2.inverse_transform(y_pred)
    y_pred_naive = scaler2.inverse_transform(y_pred_naive)
    y_test_val = scaler2.inverse_transform(y_test_val)

    smape_score = smape(y_test_val.flatten(), y_pred.flatten())
    mae_score = mean_absolute_error(y_test_val.flatten(), y_pred.flatten())
    dae_score = dae(y_test_val, y_pred)
    rmae_score = rmae(y_test_val, y_pred, y_pred_naive)

    naiv_smape = smape(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_mae = mean_absolute_error(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_dae =  dae(y_test_val, y_pred_naive)
    
    return [country, period, smape_score, mae_score, dae_score, rmae_score, naiv_smape, naiv_mae, naiv_dae, str(study.best_params)]

# Training and evaluation

In [ ]:
country_name_list = sorted(os.listdir(os.path.join("cut_data")))

final_results = {}

for country in tqdm(country_name_list):
    for period in [0,4]:
        results = get_best_params_and_test(country, period, n_trials=50)
        final_results[f"{country}_{period}"] = results
        # save as json
        with open('cb_final_results.json', 'w') as f:
            json.dump(final_results, f)
    #print(f'{country} is done')

 33%|███▎      | 1/3 [28:49<57:38, 1729.39s/it]

['Hungary', 4, 35.63163979767892, 32.92148334671887, 19.541723491717015, 1.087376215091358, 40.86548279693166, 30.276074545140673, 23.56935807641089, "{'depth': 8, 'learning_rate': 0.036492575923668544, 'iterations': 163, 'l2_leaf_reg': 9}"]
['Latvia', 0, 34.572494189699015, 9.183176635272641, 6.460148705902266, 0.8488366264834106, 39.07358328770266, 10.818544286097808, 8.088780341121478, "{'depth': 8, 'learning_rate': 0.020575221620474593, 'iterations': 491, 'l2_leaf_reg': 1}"]


 67%|██████▋   | 2/3 [1:13:04<37:53, 2273.89s/it]

['Latvia', 4, 52.31077970311959, 38.06045929868281, 25.467782987463416, 0.956844571005206, 60.14701651276256, 39.777055179085856, 29.978124138215204, "{'depth': 7, 'learning_rate': 0.04125871592650246, 'iterations': 296, 'l2_leaf_reg': 5}"]
['Poland', 0, 12.997457249986926, 5.92079684037334, 3.8666801146117318, 0.8774251373302935, 15.294045451598562, 6.747922516088737, 5.3660229914653605, "{'depth': 4, 'learning_rate': 0.02802638593499861, 'iterations': 406, 'l2_leaf_reg': 2}"]


100%|██████████| 3/3 [1:52:35<00:00, 2251.77s/it]

['Poland', 4, 27.858047120363363, 22.35726769114619, 14.529767915039427, 0.9165140495350512, 34.491730195763665, 24.393807931791184, 18.83639139483936, "{'depth': 5, 'learning_rate': 0.015582444500505659, 'iterations': 468, 'l2_leaf_reg': 9}"]


## Save results as csv file

In [ ]:
with open('cb_final_results.json', 'r') as f:
    final_results = json.load(f)
    
final_results_df = pd.DataFrame(final_results).T
final_results_df.columns = ['country', 'period', 'smape', 'mae', 'dae', 'rmae', 'naive_smape', 'naive_mae', 'naive_dae', 'best_params']
final_results_df.to_csv('cb_final_results.csv', index=False)